# VietMedBridge — 02: Nhập cả batch crawl → source + parent/child chunks

**Runtime CPU, Run all.** `EXTERNAL_SOURCE` có thể là một file `.tar` hoặc
thư mục batch chứa nhiều file `vibiomir_shard_*.tar`. Notebook nhập hết các
archive trong thư mục theo thứ tự, mỗi archive có checkpoint/build riêng.
Để tiếp tục sau restart, Run all lại với cùng batch folder.
Tên build tự sinh ổn định theo từng archive; `BUILD_RUN` chỉ cần khi muốn
đặt prefix riêng cho batch.
Hỗ trợ `vibiomir_shard_*.tar`, thư mục đã giải nén và `.tar.parts`.
Frontier bên trong tar có thể mang số shard khác `00000`.
Bản chia phần được tự ghép trên ổ local Colab và kiểm SHA-256 đúng bản gốc.
Tái sử dụng văn bản EXTRACT_SUCCESS; không tải lại các website.
Archive và snapshot BTC phải cùng SHA-256. Map bằng official URL, giữ toàn bộ
official ID/alias và một outcome cho từng ID. ID thiếu dòng crawl/extraction
được ghi thành failure với reason rõ ràng và đưa vào `external_coverage_gaps.csv`
để truy hồi sau; URL mapping sai vẫn chặn import và có báo cáo riêng.
Nguồn lỗi/challenge/redirect về trang chủ được ghi failures; LOW quality được giữ kèm flags.

Mặc định trỏ tới batch mới `data/incoming/team-crawl-archives-2026-10-08`.
Có thể điền đường dẫn thư mục batch khác; không điền lại một archive cũ.
Sau khi ghép (nếu cần), copy bốn file metadata
từ archive sang ổ local Colab để đọc; raw archive vẫn giữ làm nguồn kiểm tra.
Metadata binding và exact-source spans không chứng nhận chất lượng y khoa.

Ghi theo shard 2.048 ID, checkpoint trên Drive. Khi ngắt, Run all với cùng
batch folder/config để resume từng archive và từng shard. Một runtime ghi một batch.
Không tạo embedding, không cần GPU. Sau COMPLETE, chạy notebook 03.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "data_processing_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
DATA_WORKFLOW_API = "external-extraction-import-v6-multi-archive-batches"
upgrade = lock.get("workflow_api") != DATA_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": DATA_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "data_processing_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chọn nguồn và tokenizer


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.external_import import (
    external_build_run_name, import_external_corpus, list_external_source_batches,
)

INPUT_KIND = "external" #@param ["external", "crawl"]
EXTERNAL_SOURCE = str(DATA_ROOT / "incoming/team-crawl-archives-2026-10-08") #@param {type:"string"}
BUILD_RUN = "" #@param {type:"string"}
CRAWL_RUN = "stage-a-v2"  # chỉ dùng với INPUT_KIND="crawl"
SHARD_SIZE = 2048
MAX_NEW_SHARDS = None  # None = xử lý hết; giữ cùng config khi resume

SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
tokenizer_lock_path = DATA_ROOT / "tokenizer_lock.json"
lock = read_json(tokenizer_lock_path) if tokenizer_lock_path.exists() else {}
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(lock.get("revision"))
if not lock:
    atomic_json(tokenizer_lock_path, TOKENIZER_SPEC)
CHUNK_CONFIG = ChunkConfig(**PIPELINE_CONFIG["chunking"])
if INPUT_KIND == "external":
    if EXTERNAL_SOURCE.strip() and not Path(EXTERNAL_SOURCE).exists():
        raise FileNotFoundError(f"Không tìm thấy batch folder/archive: {EXTERNAL_SOURCE}")
    EXTERNAL_SOURCES = list_external_source_batches(
        DATA_ROOT, EXTERNAL_SOURCE.strip() or None)
    if not EXTERNAL_SOURCES:
        raise FileNotFoundError(
            "Không tìm thấy archive trong folder. EXTERNAL_SOURCE phải là file .tar "
            "hoặc folder chứa các archive .tar.")
    if BUILD_RUN.strip() and len(EXTERNAL_SOURCES) == 1:
        BUILD_RUNS = [BUILD_RUN.strip()]
    else:
        import re
        prefix = re.sub(r"[^A-Za-z0-9_.-]+", "-", BUILD_RUN.strip()).strip("-_.")
        BUILD_RUNS = []
        for source in EXTERNAL_SOURCES:
            auto_name = external_build_run_name(
                source, tokenizer_spec=TOKENIZER_SPEC, chunking=CHUNK_CONFIG,
                shard_size=SHARD_SIZE)
            if prefix:
                label = re.sub(r"[^A-Za-z0-9_.-]+", "-", Path(source).stem).strip("-_.")[:48]
                auto_name = f"{prefix}-{label}-{auto_name.rsplit('-', 1)[-1]}"
            BUILD_RUNS.append(auto_name)
    print(f"Batch sources: {len(EXTERNAL_SOURCES)} archive(s)")
    print(f"Total archive bytes: {sum(path.stat().st_size for path in EXTERNAL_SOURCES if path.is_file()):,}")
    for source, run_name in zip(EXTERNAL_SOURCES, BUILD_RUNS):
        size = source.stat().st_size if source.is_file() else 0
        print(f"  {source.name} | {size:,} bytes | build={run_name}")
elif INPUT_KIND != "crawl":
    raise ValueError("INPUT_KIND phải là external hoặc crawl.")
elif not BUILD_RUN.strip():
    BUILD_RUN = "team-100k-data-v1"
if INPUT_KIND == "crawl":
    print("Build:", BUILD_RUN, "| tokenizer:", TOKENIZER_SPEC)


## 2. Nhập/chia chunk theo shard — CPU


In [ ]:
from vietmedbridge.artifacts import utc_now
BATCH_PATH = DATA_ROOT / "active_data_batch.json"
previous_batch = read_json(BATCH_PATH) if BATCH_PATH.exists() else {}
previous_by_key = {
    (item.get("source_path"), item.get("build_run")): item
    for item in previous_batch.get("builds", [])
}
if INPUT_KIND == "external":
    batch_builds = []
    for source, run_name in zip(EXTERNAL_SOURCES, BUILD_RUNS):
        key = (str(source), run_name)
        prior = previous_by_key.get(key, {})
        batch_builds.append({**prior, "source_path": str(source), "build_run": run_name,
                             "state": "PENDING"})
    BATCH = {"schema_version": 1, "input_kind": "external", "state": "IMPORTING",
             "builds": batch_builds, "updated_at": None}
    atomic_json(BATCH_PATH, BATCH)
    for index, item in enumerate(BATCH["builds"], 1):
        item["state"] = "IMPORTING"
        atomic_json(BATCH_PATH, BATCH)
        BUILD = import_external_corpus(
            Path(item["source_path"]), OFFICIAL_LINKS, DATA_ROOT / "processed", TOKENIZER, TOKENIZER_SPEC,
            run_name=item["build_run"], config=CHUNK_CONFIG, shard_size=SHARD_SIZE,
            max_shards=MAX_NEW_SHARDS, work_dir=WORK_DIR)
        item.update({"state": "COMPLETE" if BUILD["selected_range_complete"] else "INCOMPLETE",
            "signature": BUILD["signature"], "snapshot_sha256": BUILD["snapshot_sha256"],
            "requested_input_records": BUILD["requested_input_records"], "counts": BUILD["counts"]})
        atomic_json(BATCH_PATH, BATCH)
        print(f"Archive {index}/{len(BATCH['builds'])}: {item['source_path']} -> {item['state']}")
        if not BUILD["selected_range_complete"]:
            raise RuntimeError("Batch chưa đủ shard. Run all lại với cùng folder để resume.")
    BATCH["state"] = "COMPLETE"
    BATCH["updated_at"] = utc_now()
    atomic_json(BATCH_PATH, BATCH)
    BUILDS = [{"build_run": item["build_run"], "build": read_json(
        DATA_ROOT / "processed" / item["build_run"] / "build.json")} for item in BATCH["builds"]]
    BUILD_RUN = BUILDS[-1]["build_run"]
    BUILD = BUILDS[-1]["build"]
else:
    from vietmedbridge.build import build_corpus
    BUILD = build_corpus(DATA_ROOT / "crawl" / CRAWL_RUN, DATA_ROOT / "processed",
        TOKENIZER, TOKENIZER_SPEC, run_name=BUILD_RUN, config=CHUNK_CONFIG,
        max_shards=MAX_NEW_SHARDS, official_links=OFFICIAL_LINKS, work_dir=WORK_DIR)
    BUILDS = [{"build_run": BUILD_RUN, "build": BUILD}]
    BATCH = {"schema_version": 1, "input_kind": "crawl", "state":
        "COMPLETE" if BUILD["selected_range_complete"] else "INCOMPLETE",
        "builds": [{"build_run": BUILD_RUN, "state": "COMPLETE" if BUILD["selected_range_complete"] else "INCOMPLETE",
                    "signature": BUILD["signature"], "snapshot_sha256": BUILD["snapshot_sha256"]}]}
    atomic_json(BATCH_PATH, BATCH)
BUILD_DIR = DATA_ROOT / "processed" / BUILD_RUN
TOTAL_COUNTS = {key: sum(item["build"].get("counts", {}).get(key, 0) for item in BUILDS)
                for key in sorted({name for item in BUILDS for name in item["build"].get("counts", {})})}
print(json.dumps({"state": BATCH["state"], "archives": len(BUILDS),
    "requested_input_records": sum(item["build"].get("requested_input_records", 0) for item in BUILDS),
    "counts": TOTAL_COUNTS, "build_runs": [item["build_run"] for item in BUILDS]},
    ensure_ascii=False, indent=2))
for item in BUILDS:
    build_dir = DATA_ROOT / "processed" / item["build_run"]
    source_audit = item["build"].get("source_audit")
    if source_audit:
        print(item["build_run"], "input audit:", source_audit["state"])
        print("Coverage gaps:", json.dumps(source_audit["coverage_gaps"], ensure_ascii=False))
        gap_artifact = source_audit["coverage_gaps"]["artifact"]
        if gap_artifact:
            print("Gap review file:", build_dir / gap_artifact["path"])
if BATCH["state"] == "COMPLETE":
    atomic_json(DATA_ROOT / "active_data_build.json", {
        "build_run": BUILD_RUN, "crawl_run": BUILD.get("crawl_run"),
        "signature": BUILD["signature"], "snapshot_sha256": BUILD["snapshot_sha256"],
        "input_kind": BUILD.get("input_kind", "CRAWL"), "state": BUILD["state"]})
    print("Hoàn tất toàn batch. Chạy notebook 03 trên CPU.")
else:
    raise RuntimeError("Input range chưa hoàn tất. Run all cùng config để resume.")


## 3. Kiểm tra một tài liệu và thời gian thực đo


In [ ]:
import pyarrow.parquet as pq
from vietmedbridge.build import verify_spans
for build_item in BUILDS:
    build_dir = DATA_ROOT / "processed" / build_item["build_run"]
    current_build = build_item["build"]
    found_sample = False
    for part in current_build["parts"]:
        path = build_dir / part["files"]["documents"]["path"]
        if pq.ParquetFile(path).metadata.num_rows == 0:
            continue
        doc = next(pq.ParquetFile(path).iter_batches(batch_size=1)).to_pylist()[0]
        children = pq.read_table(build_dir / part["files"]["children"]["path"], filters=[("doc_id", "=", doc["doc_id"])]).to_pylist()
        parents = pq.read_table(build_dir / part["files"]["parents"]["path"], filters=[("doc_id", "=", doc["doc_id"])]).to_pylist()
        verify_spans(doc, children, parents)
        print("Source:", build_item["build_run"], doc["doc_id"], doc["title"], "| children:", len(children))
        print(children[0]["text"][:800] if children else "NO CHILDREN")
        found_sample = True
        break
    timing = build_dir / "import_runtime.json"
    if timing.exists():
        print(build_item["build_run"], json.dumps(read_json(timing), ensure_ascii=False, indent=2))
    if not found_sample:
        print(build_item["build_run"], "no parsed document in sample scan")
print("Failures được bảo toàn trong các shard, cùng ID BTC.")


Outcome đủ range bao gồm nguồn lỗi, không có nghĩa 100% URL đã có nội dung tốt.
Notebook 03 kiểm toàn snapshot, freeze candidate và chuẩn bị các phần input
embedding trên disk. Notebook 04 sẽ đo tài nguyên trên mẫu nhỏ trước corpus lớn.
